# Triggering

Vous pouvez déclencher l'acquisition en utilisant une source extérieure (signal TTL).

Il existe trois modes de déclenchement de l'acquisition (argument `trigger_start` de la méthode `start()`):
- mode "soft" (le mode utilisé par défaut) : l'acquisition est déclenchée par le logiciel;
- mode "trig1" : l'acquisition est mise en attente de réception d'un signal de déclenchement externe sur le port `TRIG` ou `TRIG 1` (modèles >= Mµ128) du concentrateur;
- mode "trig2" : l'acquisition est mise en attente de réception d'un signal de déclenchement externe sur le port `TRIG 2` du concentrateur (modèles > = Mµ128).

Le mode de déclenchement externe accepte 4 paramètres spécifiés par `trigger_start_mode` (par défaut `"rising"`):
- "rising": déclenchement sur front montant du signal de déclenchement
- "falling": déclenchement sur front descendant du signal de déclenchement
- "high": déclenchement sur valeur haute du signal de déclenchement
- "low": déclenchement sur valeur basse du signal de déclenchement

Nous vous proposons dans ce Notebook des exemples de programmation d'acquisitions avec déclenchement.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from megamicros.log import log
from megamicros import Megamicros, AcquisitionConfig

# Set log level to INFO to see device detection details
log.setLevel("INFO")

antenna = Megamicros()

## Programmation du déclenchement

Dans l'exemple c-dessous nous plaçons l'acquisition en mode "trig1/rising":

In [ ]:
# Create configuration object
config = AcquisitionConfig(
    mems=[0, 1, 2, 3],
    sampling_frequency=50000,
    frame_length=1024,
    duration=1.0,
    trigger_start='soft', 
    trigger_start_mode='rising'
)

# Use configuration (unpack dict)
antenna.run(**config.__dict__)

# Collect signal
frames = []
for frame in antenna:
    frames.append(frame)

antenna.wait()
antenna.stop()

# Concatenate all frames
signal = np.concatenate(frames, axis=1)[1:,:]  # Skip counter row

time = np.arange(signal[0,:].shape[0]) / antenna.sampling_frequency

plt.figure(figsize=(12, 6))
for i in range(signal.shape[0]):
    plt.plot(time, signal[i,:] + i*0.5, label=f'MEMS {i}')  # Offset for visibility
plt.title('Acquired Signal from All MEMS')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude (offset for visibility)')
plt.legend()
plt.grid()
plt.show()



## Triggering with usb source

Same as before using the UsbSource low level library (see Notebook 02).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from megamicros.log import log
from megamicros.sources import UsbDataSource
from megamicros.core.config import AcquisitionConfig, UsbConfig

# Set log level to INFO to see device detection details
log.setLevel("INFO")

VENDOR_ID = 0xFE27
PRODUCT_ID = 0xAC03  # Example: Mu32-usb3 / Mu256-usb3 (0xAC01)

usb_config = UsbConfig(vendor_id=VENDOR_ID, product_id=PRODUCT_ID)
source = UsbDataSource(usb_config)

In [ ]:

# Create configuration object
config = AcquisitionConfig(
    mems=[0, 1, 2, 3],
    sampling_frequency=50000,
    frame_length=1024,
    duration=5.0,
    timeout=1,
    time_activation=200,
    trigger_start='trig1', 
    trigger_start_mode='rising'
)

try: 
    frames = []
    source.configure(config)
    source.start()
            
    # Collect frames in a list    
    for frame in source:
        frames.append(frame)

    source.wait()
    source.stop()

    # Concatenate all frames
    signal = np.concatenate(frames, axis=1)[1:,:]  # Skip counter row

    time = np.arange(signal[0,:].shape[0]) / config.sampling_frequency

    plt.figure(figsize=(12, 6))
    for i in range(signal.shape[0]):
        plt.plot(time, signal[i,:] + i*0.5, label=f'MEMS {i}')  # Offset for visibility
    plt.title('Acquired Signal from All MEMS')
    plt.xlabel('Time (s)')
    plt.ylabel('Amplitude (offset for visibility)')
    plt.legend()
    plt.grid()
    plt.show()

except Exception as e:
    print(f"Error during acquisition: {e}")